##정형
핵심 라이브러리: transformers(BERT 모델용),  xgboost(최종 분류기용)
- 비정형 데이터(임상 노트)와 정형 데이터(활력 징후, 랩 수치 등)를 환자 입원 ID(hadm_id) 기준으로 Inner Join(교집합) 하여 결합, 모델은 환자의 '상태'와 '기록'
  
- 텍스트 전처리 핵심 포인트: 퇴원 시점의 정보(사망 시간, 퇴원 장소 등)를 모두 배제하고, 오직 입원 초기(Admission) 섹션인 '주호소', '현병력', '신체검진' 등만 정규표현식으로 추출

- ClinicalBERT 임베딩 추출 (Chunking 기법)
설명: 텍스트를 AI가 이해할 수 있는 숫자 벡터로 변환, BERT의 입력 길이 제한(512토큰)을 극복하기 위해 Chunking(조각내기) 기법을 사용. 긴 문서를 여러 조각으로 나누어 임베딩한 뒤 평균을 내어, 정보 손실을 최소화

- 정형 데이터 전처리 및 피처 결합 (Late Fusion)
정형 데이터의 결측치를 채우고 스케일링한 뒤, BERT 벡터와 합치는 과정, 핵심 포인트: Late Fusion 방식을 사용하여 텍스트 벡터(768차원) + 정형 벡터(N차원)를 하나로 합칩

- XGBoost 학습 및 평가
결합된 데이터를 강력한 분류기인 XGBoost로 학습시킵, scale_pos_weight 파라미터를 사용하여 재입원 데이터(소수 클래스) 불균형 문제를 해결

In [ ]:
import pandas as pd
import numpy as np
import torch
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# ==========================================
# [설정] 경로 및 파라미터
# ==========================================
CONFIG = {
    # 1. 텍스트 데이터 (기존)
    "discharge_file": "data/physionet.org/files/mimic-iv-note/2.2/note/discharge.csv", 
    # 2. 정형 데이터 (새로 추가된 파일)
    "structured_file": "modeling_dataset.csv", 
    
    "model_path": "./bio_clinicalbert",
    "n_samples": 20000,  # 전체 데이터 사용 시 None 또는 큰 숫자
    "random_seed": 42,
    "chunk_limit": 4     # 텍스트 청킹 개수
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">>> 사용 장치: {device}")

# ==========================================
# 1. 데이터 로드 및 병합
# ==========================================
def load_hybrid_data():
    print(">>> 1. 데이터 로드 및 병합 중...")
    
    # (1) 텍스트 데이터 로드
    df_text = pd.read_csv(CONFIG["discharge_file"])
    
    # (2) 정형 데이터 로드 (제공해주신 데이터)
    # hadm_id를 기준으로 병합해야 하므로 문자열/정수형 타입 일치 주의
    df_struc = pd.read_csv(CONFIG["structured_file"])
    
    # (3) 병합 (Inner Join: 텍스트와 정형 데이터가 모두 있는 경우만)
    # hadm_id 기준으로 합칩니다.
    df = pd.merge(df_text, df_struc, on='hadm_id', how='inner')
    
    # (4) 타겟 변수 확인 (readmission_30d)
    if 'readmission_30d' not in df.columns:
        raise ValueError("정형 데이터에 'readmission_30d' 컬럼이 없습니다.")
        
    # (5) 샘플링 (데이터가 너무 많을 경우)
    if len(df) > CONFIG["n_samples"]:
        df = df.sample(n=CONFIG["n_samples"], random_state=CONFIG["random_seed"]).copy()
        
    print(f"   - 최종 데이터 수: {len(df)}건")
    print(f"   - 재입원 비율: {df['readmission_30d'].mean()*100:.2f}%")
    
    return df

# ==========================================
# 2. 텍스트 처리 (BERT Embedding)
# ==========================================
# (이전 코드와 동일한 White-list 전처리 및 Chunking 로직)
def extract_admission_sections(text):
    if pd.isna(text): return ""
    text = str(text)
    patterns = [
        r"Chief Complaint:\s*(.*?)(?=\n[A-Z])",
        r"History of Present Illness:\s*(.*?)(?=\n[A-Z])",
        r"Past Medical History:\s*(.*?)(?=\n[A-Z])",
        r"Physical Exam:\s*(.*?)(?=\n[A-Z])"
    ]
    extracted = []
    for pat in patterns:
        match = re.search(pat, text, re.DOTALL | re.IGNORECASE)
        if match and len(match.group(1)) > 5:
            extracted.append(match.group(1).strip())
    if not extracted: return text[:512]
    return " ".join(extracted)

class ClinicalBERTChunker:
    def __init__(self):
        print("   - BERT 모델 로드 중...")
        self.tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_path'])
        self.model = AutoModel.from_pretrained(CONFIG['model_path']).to(device)
        
    def get_embedding(self, text):
        tokens = self.tokenizer(text, add_special_tokens=False, return_tensors='pt')['input_ids'][0]
        chunk_embeddings = []
        stride = 510
        
        for i in range(0, len(tokens), stride):
            chunk = tokens[i : i + stride]
            input_ids = torch.cat([
                torch.tensor([self.tokenizer.cls_token_id]), chunk, torch.tensor([self.tokenizer.sep_token_id])
            ]).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = self.model(input_ids=input_ids)
                chunk_embeddings.append(output.last_hidden_state[:, 0, :].cpu().numpy())
            if len(chunk_embeddings) >= CONFIG['chunk_limit']: break
            
        if not chunk_embeddings: return np.zeros(768)
        return np.mean(chunk_embeddings, axis=0)

    def process(self, texts):
        self.model.eval()
        return np.vstack([self.get_embedding(t) for t in tqdm(texts, desc="BERT Embedding")])

# ==========================================
# 3. 정형 데이터 전처리 (Preprocessing)
# ==========================================
def preprocess_structured_data(df):
    print(">>> 3. 정형 데이터 전처리 중...")
    
    # 학습에 사용할 정형 데이터 컬럼 선정
    # (ID, 타겟, 텍스트, 날짜 관련 등 불필요한 컬럼 제외)
    exclude_cols = ['hadm_id', 'readmission_30d', 'text', 'clean_text', 
                    'admit_hour', 'admit_day_of_week', 'admit_month', # 시간 정보는 제외하거나 인코딩 필요
                    'subject_id', 'charttime', 'storetime'] # 메타데이터 제외
    
    feature_cols = [c for c in df.columns if c not in exclude_cols and df[c].dtype in ['int64', 'float64']]
    
    print(f"   - 사용된 정형 피처 수: {len(feature_cols)}개")
    
    # 파이프라인 구축: 결측치 평균 대치 -> 스케일링(StandardScaler)
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')), # 결측치는 평균값으로 채움
        ('scaler', StandardScaler())                 # 정규화 (평균0, 분산1)
    ])
    
    X_struc = pipeline.fit_transform(df[feature_cols])
    return X_struc, feature_cols

# ==========================================
# 4. 메인 실행
# ==========================================
if __name__ == "__main__":
    try:
        # 1. 데이터 로드
        df = load_hybrid_data()
        
        # 2. 텍스트 피처 추출 (BERT)
        print(">>> 2. 텍스트 처리 중...")
        df['clean_text'] = df['text'].apply(extract_admission_sections)
        bert_extractor = ClinicalBERTChunker()
        X_text = bert_extractor.process(df['clean_text'].tolist()) # (N, 768)
        
        # 3. 정형 피처 추출
        X_struc, feature_names = preprocess_structured_data(df)    # (N, M)
        
        # 4. 피처 결합 (Hybrid)
        print(">>> 4. 피처 결합 (Text + Structured)...")
        X_final = np.hstack([X_text, X_struc])
        y = df['readmission_30d'].values
        
        print(f"   - 최종 입력 벡터 크기: {X_final.shape}")
        
        # 5. 학습 및 평가
        X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, stratify=y, random_state=42)
        
        print(">>> 5. XGBoost 학습 중...")
        scale_pos = (len(y) - sum(y)) / sum(y) # 불균형 가중치
        
        xgb = XGBClassifier(
            n_estimators=500,
            learning_rate=0.02,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos,
            n_jobs=-1,
            random_state=42,
            early_stopping_rounds=50
        )
        
        xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        
        # 결과 출력
        y_prob = xgb.predict_proba(X_test)[:, 1]
        y_pred = xgb.predict(X_test)
        
        auc_score = roc_auc_score(y_test, y_prob)
        
        print("\n" + "="*50)
        print(f" [Hybrid Model 성능 결과]")
        print("="*50)
        print(f" AUROC Score: {auc_score:.4f}")
        print("-" * 50)
        print(classification_report(y_test, y_pred))
        
        # 중요도 분석 (정형 데이터 중 무엇이 중요한지)
        # BERT 피처(0~767) 뒤에 정형 피처가 붙어있음
        importances = xgb.feature_importances_
        struc_importances = importances[768:] # 정형 데이터 중요도만 추출
        
        if len(feature_names) == len(struc_importances):
            print("\n[주요 영향 변수 Top 10]")
            top_indices = np.argsort(struc_importances)[::-1][:10]
            for idx in top_indices:
                print(f" - {feature_names[idx]}: {struc_importances[idx]:.4f}")
                
    except Exception as e:
        print(f"오류 발생: {e}")

## 자연어 처리만 (discharge.csv 파일만 사용해서)


In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# ==========================================
# 1. 데이터 준비 및 라벨링 (30일 재입원)
# ==========================================
def load_and_prep_data(discharge_path, admissions_path):
    print(">>> 1. 데이터 로드 및 병합 중...")
    # 1. 데이터 로드
    df_notes = pd.read_csv(discharge_path)
    df_adm = pd.read_csv(admissions_path, parse_dates=['admittime', 'dischtime'])
    
    # 2. 병합 (Note + Admission Info)
    # hadm_id가 있는 노트만 사용
    df = pd.merge(df_notes, df_adm[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'hospital_expire_flag']], 
                  on=['subject_id', 'hadm_id'], how='inner')
    
    # 3. 사망 환자 제외 (재입원 예측의 논리적 타당성 확보)
    # 병원에서 사망한 환자는 재입원 할 수 없으므로 데이터셋에서 제외해야 함
    df = df[df['hospital_expire_flag'] == 0].copy()
    
    # 4. 30일 이내 재입원 라벨 생성
    df = df.sort_values(by=['subject_id', 'dischtime'])
    df['next_admittime'] = df.groupby('subject_id')['admittime'].shift(-1)
    df['days_to_next'] = (df['next_admittime'] - df['dischtime']).dt.total_seconds() / (24 * 3600)
    
    # 라벨: 30일 이내(1), 그 외(0)
    df['label'] = df['days_to_next'].apply(lambda x: 1 if (not pd.isna(x) and x <= 30 and x >= 0) else 0)
    
    print(f" - 분석 대상 데이터 수: {len(df)}건")
    print(f" - 재입원 비율: {df['label'].mean()*100:.2f}%")
    
    return df

# ==========================================
# 2. 텍스트 전처리: 입원 초기 섹션 추출 (White-list)
# ==========================================
def extract_admission_sections(text):
    """
    퇴원 요약지 전체가 아니라, 입원 시점에 작성된 '안전한' 섹션만 추출합니다.
    이 방법은 논문의 '데이터 유출' 문제를 근본적으로 해결합니다.
    """
    if pd.isna(text): return ""
    text = str(text)
    
    # 추출할 섹션 헤더들 (입원 초기 정보)
    # MIMIC 노트 스타일에 맞춰 정규식 패턴 정의
    target_patterns = [
        r"Chief Complaint:\s*(.*?)(?=\n[A-Z])",  # 주호소
        r"History of Present Illness:\s*(.*?)(?=\n[A-Z])", # 현병력
        r"Past Medical History:\s*(.*?)(?=\n[A-Z])", # 과거력
        r"Physical Exam:\s*(.*?)(?=\n[A-Z])", # 신체검진
        r"Pertinent Results:\s*(.*?)(?=\n[A-Z])" # 주요 검사 결과 (입원 당시)
    ]
    
    extracted_content = []
    for pattern in target_patterns:
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if match:
            # 너무 짧은 섹션(없음, denied 등)은 제외 가능
            content = match.group(1).strip()
            if len(content) > 5: 
                extracted_content.append(content)
    
    # 추출된 내용이 없으면 원본의 앞부분 512자 사용 (Fallback)
    if not extracted_content:
        return text[:512]
        
    return " ".join(extracted_content)

# ==========================================
# 3. ClinicalBERT 임베딩 추출 (핵심)
# ==========================================
class ClinicalBERTFeatureExtractor:
    def __init__(self, model_name="emilyalsentzer/Bio_ClinicalBERT", batch_size=32):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f">>> 3. ClinicalBERT 로드 중... (Device: {self.device})")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.batch_size = batch_size

    def extract_features(self, texts):
        self.model.eval()
        all_embeddings = []
        
        # Progress Bar와 함께 배치 처리
        for i in tqdm(range(0, len(texts), self.batch_size), desc="Embedding Extraction"):
            batch_texts = texts[i : i + self.batch_size]
            
            # 토큰화 (최대 길이 512)
            inputs = self.tokenizer(batch_texts, padding=True, truncation=True, 
                                    max_length=512, return_tensors="pt").to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            # [CLS] 토큰의 벡터를 문장 대표 벡터로 사용
            # outputs.last_hidden_state shape: (batch, seq_len, 768)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.extend(cls_embeddings)
            
        return np.array(all_embeddings)

# ==========================================
# 4. 메인 실행 코드
# ==========================================
# 파일 경로 설정 (사용자 환경에 맞게 수정)
DISCHARGE_FILE = 'data/physionet.org/files/mimic-iv-note/2.2/note/discharge.csv' 
ADMISSIONS_FILE = 'data/physionet.org/files/mimiciv/3.1/hosp/admissions.csv'

# 1) 데이터 로드
try:
    df = load_and_prep_data(DISCHARGE_FILE, ADMISSIONS_FILE)
    
    # 시간 관계상 샘플링 (전체 데이터 사용 시 이 줄 삭제)
    # GPU가 없으면 2000~5000건 정도로 테스트 권장
    df_sample = df.sample(n=3000, random_state=42).copy() 
    print(f" -> 학습 속도를 위해 {len(df_sample)}건 샘플링하여 진행합니다.")

    # 2) 텍스트 전처리 (유출 방지 섹션 추출)
    print(">>> 2. 입원 초기 섹션 추출 중 (Data Leakage 방지)...")
    df_sample['clean_text'] = df_sample['text'].apply(extract_admission_sections)
    
    # 3) ClinicalBERT 임베딩 생성
    extractor = ClinicalBERTFeatureExtractor(model_name="./bio_clinicalbert", batch_size=16)
    X_vectors = extractor.extract_features(df_sample['clean_text'].tolist())
    y_labels = df_sample['label'].values
    
    # 4) 모델 학습 및 평가 (Logistic Regression on BERT Embeddings)
    print(">>> 4. 예측 모델 학습 및 평가...")
    X_train, X_test, y_train, y_test = train_test_split(X_vectors, y_labels, test_size=0.2, random_state=42, stratify=y_labels)
    
    # class_weight='balanced': 재입원 데이터 불균형 해소
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_train, y_train)
    
    # 결과 출력
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_prob)
    
    print("\n" + "="*40)
    print(f"   [최종 모델 성능 평가: ClinicalBERT + LR]")
    print("="*40)
    print(f"AUROC Score: {auc:.4f}")
    print("-" * 40)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("-" * 40)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
except FileNotFoundError:
    print("[오류] CSV 파일을 찾을 수 없습니다. 파일 경로를 확인해주세요.")
except Exception as e:
    print(f"[오류] 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
# ==========================================
# [설정] 하이퍼파라미터 및 경로
# ==========================================
CONFIG = {
    # 여기에 사용자님이 알려주신 절대 경로를 그대로 넣습니다.
    "discharge_file": "data/physionet.org/files/mimic-iv-note/2.2/note/discharge.csv",
    "admissions_file": "data/physionet.org/files/mimiciv/3.1/hosp/admissions.csv",
    
    "model_path": "./bio_clinicalbert", # 모델 폴더는 medi.py와 같은 위치에 있어야 합니다
    "n_samples": 20000,                 
    "batch_size": 16,                   
    "random_seed": 42,
    "max_bert_len": 512,
    "chunk_limit": 4                    
}

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">>> 사용 장치(Device): {device}")

# ==========================================
# 1. 데이터 로드 및 라벨링
# ==========================================
def load_data():
    print(f">>> 1. 데이터 로드 중... (Target: {CONFIG['n_samples']}건)")
    
    # 1. CSV 로드
    df_notes = pd.read_csv(CONFIG["discharge_file"])
    df_adm = pd.read_csv(CONFIG["admissions_file"], parse_dates=['admittime', 'dischtime'])
    
    # 2. 병합
    df = pd.merge(df_notes, df_adm[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'hospital_expire_flag']], 
                  on=['subject_id', 'hadm_id'], how='inner')
    
    # 3. 병원 내 사망자 제외 (재입원 예측 타당성 확보)
    df = df[df['hospital_expire_flag'] == 0].copy()
    
    # 4. 30일 재입원 라벨 생성
    df = df.sort_values(by=['subject_id', 'dischtime'])
    df['next_admittime'] = df.groupby('subject_id')['admittime'].shift(-1)
    df['days_to_next'] = (df['next_admittime'] - df['dischtime']).dt.total_seconds() / (24 * 3600)
    df['label'] = df['days_to_next'].apply(lambda x: 1 if (not pd.isna(x) and x <= 30 and x >= 0) else 0)
    
    # 5. 샘플링 (재입원 비율 유지하며 추출)
    if len(df) > CONFIG["n_samples"]:
        # Positive/Negative 비율 맞춰서 샘플링
        df = df.groupby('label', group_keys=False).apply(
            lambda x: x.sample(int(np.rint(CONFIG["n_samples"]*len(x)/len(df))), random_state=CONFIG["random_seed"])
        ).sample(frac=1).reset_index(drop=True) # 셔플
        
    print(f"   - 최종 데이터 수: {len(df)}건")
    print(f"   - 재입원(Positive) 비율: {df['label'].mean()*100:.2f}%")
    return df

# ==========================================
# 2. 전처리: 입원 초기 섹션 추출 (White-list)
# ==========================================
def extract_admission_sections(text):
    if pd.isna(text): return ""
    text = str(text)
    
    # 입원 초기에 작성되는 핵심 섹션만 추출 (유출 방지)
    patterns = [
        r"Chief Complaint:\s*(.*?)(?=\n[A-Z])",
        r"History of Present Illness:\s*(.*?)(?=\n[A-Z])",
        r"Past Medical History:\s*(.*?)(?=\n[A-Z])",
        r"Physical Exam:\s*(.*?)(?=\n[A-Z])",
    ]
    
    extracted = []
    for pat in patterns:
        match = re.search(pat, text, re.DOTALL | re.IGNORECASE)
        if match and len(match.group(1)) > 5:
            extracted.append(match.group(1).strip())
            
    if not extracted:
        return text[:512] # 섹션 못 찾으면 앞부분만 사용
    
    return " ".join(extracted)

# ==========================================
# 3. ClinicalBERT Chunking 임베딩 추출
# ==========================================
class ClinicalBERTChunker:
    def __init__(self):
        print(f">>> 3. ClinicalBERT 로드 중... (경로: {CONFIG['model_path']})")
        self.tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_path'])
        self.model = AutoModel.from_pretrained(CONFIG['model_path']).to(device)
        
    def get_embedding(self, text):
        """긴 텍스트를 512 토큰씩 잘라서 평균 임베딩"""
        # 전체 토큰화
        tokens = self.tokenizer(text, add_special_tokens=False, return_tensors='pt')['input_ids'][0]
        
        # 짧은 문장은 바로 처리
        if len(tokens) <= (CONFIG['max_bert_len'] - 2):
            inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, 
                                    max_length=CONFIG['max_bert_len']).to(device)
            with torch.no_grad():
                output = self.model(**inputs)
            return output.last_hidden_state[:, 0, :].cpu().numpy()
        
        # 긴 문장 Chunking
        chunk_embeddings = []
        # 510 = 512 - 2 ([CLS], [SEP])
        stride = 510 
        
        for i in range(0, len(tokens), stride):
            chunk = tokens[i : i + stride]
            
            # [CLS] + Tokens + [SEP] 구조 생성
            input_ids = torch.cat([
                torch.tensor([self.tokenizer.cls_token_id]),
                chunk,
                torch.tensor([self.tokenizer.sep_token_id])
            ]).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = self.model(input_ids=input_ids)
                # [CLS] 벡터 추출
                chunk_embeddings.append(output.last_hidden_state[:, 0, :].cpu().numpy())
            
            # 너무 오래 걸리지 않게 Chunk 개수 제한
            if len(chunk_embeddings) >= CONFIG['chunk_limit']: 
                break
                
        # Chunk들의 평균 계산 (Mean Pooling)
        return np.mean(chunk_embeddings, axis=0)

    def process_batch(self, texts):
        self.model.eval()
        embeddings = []
        print("   - BERT 임베딩 추출 시작 (Chunking 적용)...")
        
        for text in tqdm(texts):
            emb = self.get_embedding(text)
            embeddings.append(emb)
            
        return np.vstack(embeddings)

# ==========================================
# 4. 결과 시각화 (논문용 그래프)
# ==========================================
def plot_results(y_test, y_pred, y_prob):
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))

    # 1. ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    ax[0].plot(fpr, tpr, color='#d62728', lw=2, label=f'ClinicalBERT+XGB (AUC = {roc_auc:.3f})')
    ax[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax[0].set_title('ROC Curve (30-day Readmission)', fontsize=14, fontweight='bold')
    ax[0].set_xlabel('False Positive Rate')
    ax[0].set_ylabel('True Positive Rate')
    ax[0].legend(loc="lower right")
    ax[0].grid(alpha=0.3)

    # 2. Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[1], annot_kws={"size": 14})
    ax[1].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    ax[1].set_xlabel('Predicted')
    ax[1].set_ylabel('Actual')
    ax[1].set_xticklabels(['No Readmission', 'Readmission'])
    ax[1].set_yticklabels(['No Readmission', 'Readmission'])

    plt.tight_layout()
    plt.show() # Jupyter/Colab용
    plt.savefig("result_graphs.png", dpi=300) # 파일 저장

# ==========================================
# 5. 메인 파이프라인 실행
# ==========================================
if __name__ == "__main__":
    try:
        # 1. 데이터 로드
        df = load_data()
        
        # 2. 전처리 (White-list Sectioning)
        print(">>> 2. 텍스트 전처리 (입원 초기 섹션 추출)...")
        df['clean_text'] = df['text'].apply(extract_admission_sections)
        
        # 3. 임베딩 추출
        extractor = ClinicalBERTChunker()
        X = extractor.process_batch(df['clean_text'].tolist())
        y = df['label'].values
        
        # 4. 데이터 분할
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG['random_seed'], stratify=y)
        
        # 5. XGBoost 학습
        print(">>> 4. XGBoost 모델 학습 중...")
        # scale_pos_weight: 불균형 데이터 가중치 (Negative 수 / Positive 수)
        weight = (len(y) - sum(y)) / sum(y)
        
        xgb = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            scale_pos_weight=weight,
            random_state=CONFIG['random_seed'],
            n_jobs=-1,
            eval_metric='auc',
            early_stopping_rounds=30  # 조기 종료
        )
        
        # 검증셋을 사용하여 과적합 방지
        xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        
        # 6. 결과 평가
        y_pred = xgb.predict(X_test)
        y_prob = xgb.predict_proba(X_test)[:, 1]
        
        print("\n" + "="*50)
        print(" [최종 성능 평가: ClinicalBERT(Chunking) + XGBoost]")
        print("="*50)
        print(classification_report(y_test, y_pred))
        print(f"AUROC Score: {roc_auc_score(y_test, y_prob):.4f}")
        print("="*50)
        
        # 7. 그래프 그리기
        plot_results(y_test, y_pred, y_prob)
        
    except Exception as e:
        print(f"\n[Error] 실행 중 오류가 발생했습니다: {e}")